<font color="purple"><font size="7"> OpenClassrooms Projet 3 </font>

<font color="grey"> Chargement des modules necessaires </font>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Nettoyage des données

## <font color="black"> Chargement du jeu de données</font>

In [ ]:
df=pd.read_csv("data/fr.openfoodfacts.org.products.csv",sep='\t',low_memory=False)


In [ ]:
df.shape

## Nombres et type de variable

In [ ]:
df.dtypes.value_counts()

## Valeurs manquantes

### Identification

In [ ]:
plt.figure(figsize=(20,10))
plt.title("Visualisation des données manquantes",fontweight='bold',fontsize=15)
sns.heatmap(df.isna(), cbar=False)
plt.xticks(fontsize='15')
plt.show()

### Traitement

Supprimons les variables ayant plus de 50% de valeurs manquantes

In [ ]:
bad_var=df.columns[(df.isna().sum(axis = 0)/df.shape[0]) > 0.5]

In [ ]:
df=df.drop(columns=bad_var)

In [ ]:
df.shape

In [ ]:
plt.figure(figsize=(20,10))
plt.title("Visualisation des données manquantes",fontweight='bold',fontsize=15)
sns.heatmap(df.isna(), cbar=False)
plt.xticks(fontsize='15')
plt.show()

## Doublons

### Identification

In [ ]:
df[df.duplicated()].shape

Considérons les individus ayant le meme 'product_name', le meme 'brands' et le meme 'energy_100g" sauf la permière occurence

In [ ]:
doublons_df=df[df.duplicated(subset=['product_name','brands','energy_100g'],keep='first')]

In [ ]:
doublons_df.shape

### Traitement

Enlevons les doublons du dataset

In [ ]:
df=df.drop(index=doublons_df.index)

## Selection des variables

Sortons les variables redondantes ou sans interet

In [ ]:
df=df.drop(columns=['created_t',
                 'last_modified_t',
                 'url',
                 'brands_tags',
                 'countries',
                 'countries_tags',
                 'states',
                 'states_fr',
                 'states_tags',
                 'creator'])

In [ ]:
df.shape

## Changement des types de données

In [ ]:
df.created_datetime=pd.to_datetime(df.created_datetime,errors='coerce')
df.last_modified_datetime=pd.to_datetime(df.last_modified_datetime,errors='coerce')

cols = ['code',
        'product_name',
        'brands',
        'countries_fr',
        'ingredients_text',
        'additives',
        'nutrition_grade_fr']
df[cols] = df[cols].astype(pd.StringDtype())

## Sauvegarde des données

In [ ]:
df.to_pickle("data/openfoodfact_clean")

In [ ]:
df=pd.read_pickle("data/openfoodfact_clean");


# Etude Univariée

In [ ]:
df_test=pd.read_pickle("data/openfoodfact_clean");

## Variable quantitative

In [ ]:
df_test.describe()

Suppressions des variables sans information

df_test=df_test.drop(columns=['ingredients_from_palm_oil_n','ingredients_that_may_be_from_palm_oil_n'])

### Identification et suppression des outliers

#### Valeurs impossibles

Suppression des outliers sur les variables _100g, les valeurs ne peuvent pas être supérieur à 100 ou négative

Suppression des valeurs de fibre supérieures à 60 (donnée métier)

In [ ]:
for col in [
            'fat_100g',
            'saturated-fat_100g',
            'carbohydrates_100g',
            'sugars_100g',
            'fiber_100g',
            'proteins_100g',
            'salt_100g',
            'sodium_100g'
]:
    
    print(f'suppression de {df_test[(df_test[col]>100) | (df_test[col]<0)].shape[0]} individus basé sur la colonne {col}')
    df_test=df_test.drop(index=df_test[(df_test[col]>100) | (df_test[col]<0)].index)
    

In [ ]:
df_test=df_test.drop(index=df_test[df_test['fiber_100g']>60].index)

In [ ]:
df_test.shape

#### Valeur de variable fille supérieure à celle de la variable mère

In [ ]:
df_impossible=df_test[df_test['saturated-fat_100g']>df_test['fat_100g']]
df_test.loc[df_impossible.index,"fat_100g"]=df_test.loc[df_impossible.index,"saturated-fat_100g"]

In [ ]:
df_impossible=df_test[df_test['sugars_100g']>df_test['carbohydrates_100g']]
df_test.loc[df_impossible.index,"sugars_100g"]=df_test.loc[df_impossible.index,"carbohydrates_100g"]

#### Suppression des outliers sur la regle des quantiles

Fonction identifiant et supprimant les outliers

In [ ]:
#fonction qui retourne le dataframe sans les outliers basés sur la régle des quantiles
def remove_outliers(data,col):
    Quant=data[col].quantile([0.25,0.5,0.75]).values #Calcul des quantiles
    
    if int(Quant[2])==0:
        print(f'pas de traitement car Q3=0')
        return data
    else:
        n=data[(data[col]<0) | (data[col]>=(Quant[1]+1.5*Quant[2]))].shape[0] # nombre d'outliers
        print(f'traitement de la colonne {col} suppression de {n} individus avec une valeur en dehors de l\'interval [0,{Quant[1]+1.5*Quant[2]}]')
        return data[(data[col]>=0) & (data[col]<Quant[1]+1.5*Quant[2]) | (data[col].isna())]


Applications au données

In [ ]:
df_test=remove_outliers(df_test,'energy_100g')

In [ ]:
df_test.shape

###  Boxplot

In [ ]:
i=0
colors=sns.color_palette("Set3", 14)
plt.figure(figsize=(15,15))
for col in df_test.describe():
    i+=1
    plt.subplot(5,3,i)
    sns.boxplot(x=df_test[col],color=colors[i%8])
plt.tight_layout()

### Distributions

In [ ]:
fig, ax = plt.subplots(4,3)
fig.set_size_inches(20, 20)
i=0
for col in df_test.describe().columns:
    binw=(df_test[col].max()-df_test[col].min())/20
    sns.histplot(x=df_test[col],binwidth=binw,ax=ax[i//3][i%3],color=colors[i%8])
    i+=1


## Variables qualitatives

In [ ]:
print('{:20} {:^30} {:^20}'.format('columns','unique','pourcentage manquant'))
print('-'*70)
for col in df_test.select_dtypes('string'):
    print (f'{col:20} {len(df_test[col].unique()):20} {df_test[col].isna().sum(axis = 0)/df_test.shape[0]:20.3f}')
    print('-'*70)

Nous décidons de ne pas travailler avec certaines colonnes

In [ ]:
df_test=df_test.drop(columns=['code','ingredients_text','additives'])

### Variables countries_fr

Création d'une nouvele colonne ave cles pays splitter en liste

In [ ]:
df_test['countries_list']=df['countries_fr'].str.split(',')

In [ ]:
df_test['countries_list']=df_test['countries_list'].fillna('0')

Récupération de la liste des pays

In [ ]:
def split_text(df,column,separator):
    values=set()
    for data in df[column].fillna('0'):    
        for value in str.split(data,sep=separator):
            values.add(value.lower())
    return values

In [ ]:
countries_list=list(split_text(df_test,'countries_fr',','))

In [ ]:
len(countries_list)

Ecriture de la liste dans un csv

In [ ]:
pd.DataFrame(countries_list).to_csv('data/countries_add.csv',sep=',')

Récupération et création du dictionnaire de correspondance

In [ ]:
countries=pd.read_csv('data/countries_correct.csv',sep=',',index_col=0)

In [ ]:
dict_countries={}
for i in countries.index:
    dict_countries[countries.iloc[i][0]]=countries.iloc[i][1]

Fonction de nettoyage

In [ ]:
def clean_countries(data,individu):
    new_countries=[]
    for country in data.loc[individu,'countries_list']:
        new_countries.append(dict_countries[country.lower()])
    data.at[individu,'countries_list_clean']=new_countries

Application aux données

In [ ]:
for i in df_test.index:
    clean_countries(df_test,i)

Vérification du nombre de pays différents

In [ ]:
countries_new=set()
for countries in df_test['countries_list_clean']:
    for country in countries:
        countries_new.add(country)
len(countries_new)

In [ ]:
len(set(dict_countries.values()))

Séparation de pays en deux zones

In [ ]:
df_test['in_US']=[int('états-unis' in t) for t in df_test['countries_list_clean']]

In [ ]:
df_test['in_US'].sum()

In [ ]:
pays_europe=["belgique",
                "espagne",
                "italie",
                "allemagne",
                "france",
                "portugal",
                "estonie",
                "hongrie",
                "islande",
                "suisse",
                "slovénie",
                "suéde",
            "pays-bas",
            "royaume-uni",
            "croatie",
            "slovaquie",
            "autriche",
            "écosse",
            "grèce",
            "luxembourg",
            "angleterre",
            "bulgarie",
            "finlande",
            "danemark",
            "norvège"
            "pologne"]

In [ ]:
df_test['in_EU']=[int(bool(len(set(t).intersection(pays_europe)))) for t in df_test['countries_list_clean']]

In [ ]:
df_test['in_EU'].sum()

## Sauvegarde des données

In [ ]:
df_test.to_pickle("data/data_univ")

In [ ]:
df_test=pd.read_pickle("data/data_univ")

# Analyse Multivarié

###  Corrélations

Pairs de valeurs qui semblent très corrélées

In [ ]:
from scipy.stats.stats import spearmanr

In [ ]:
def spearman(df,col1,col2):
    df_corr=df_corr=df_test.dropna(subset=[col1,col2])
    S=spearmanr(df_corr[col1],df_corr[col2])[0]
    return S
  

In [ ]:
df_test.describe().columns

In [ ]:
cols=['additives_n', 'energy_100g', 'fat_100g', 'saturated-fat_100g',
       'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g',
       'salt_100g', 'nutrition-score-fr_100g']

In [ ]:
df_corr=df_test.dropna(subset=['salt_100g','sodium_100g'])
print(df_corr.shape)
spearmanr(df_corr['salt_100g'],df_corr['sodium_100g'])

In [ ]:
df_corr=df_test.dropna(subset=['nutrition-score-fr_100g','nutrition-score-uk_100g'])
print(df_corr.shape)
spearmanr(df_corr['nutrition-score-fr_100g'],df_corr['nutrition-score-uk_100g'])

Nous pouvons donc sortir ces deux variables de l'analyse

In [ ]:
df_test=df_test.drop(columns=['nutrition-score-uk_100g','sodium_100g'])

In [ ]:
sns.pairplot(df_test[['additives_n',
                      'energy_100g',
                      'fat_100g',
                      'saturated-fat_100g',
                      'carbohydrates_100g',
                      'sugars_100g',
                      'fiber_100g',
                      'proteins_100g',
                      'salt_100g',
                      'nutrition-score-fr_100g']],plot_kws=dict(s=2))

In [ ]:

df_test.to_pickle("data/openfoodfact_clean2")

In [ ]:
df_test=pd.read_pickle("data/openfoodfact_clean2")

### Régression linéaire

#### Formule de l'energie en fonction des nutriments:



##### Formule à partir d'un régression linéaire

Création du dataset pour la régression

In [ ]:
df_incomplete=df_test[df_test['fat_100g'].isna() |
                     df_test['proteins_100g'].isna() |
                     df_test['carbohydrates_100g'].isna() |
                     df_test['energy_100g'].isna()
                     ]

In [ ]:
df_complete=df_test.drop(index=df_incomplete.index)

In [ ]:
df_complete.shape

In [ ]:
from sklearn.linear_model import LinearRegression

#créer un objet reg lin
modeleReg=LinearRegression(fit_intercept=True)

#créer y et X
y=df_complete['energy_100g']
X=df_complete[['fat_100g',
             'proteins_100g',             
             'carbohydrates_100g',
             ]]

modeleReg.fit(X,y)

In [ ]:
reg_coef=modeleReg.coef_
print(f'Linear regression coefficients are {reg_coef}')

a=np.linspace(0,1500,100)

#calcul du R²
print(f'R² is {modeleReg.score(X,y)}')
plt.figure(figsize=(10,10))
plt.plot(y, modeleReg.predict(X),'.',markersize=2)
plt.plot(a,a*4.184,color='r',linewidth=1)
plt.xlabel('Valeurs observées')
plt.ylabel('Valeurs calculées')

plt.show()

In [ ]:
plt.figure(figsize=(10,10))
plt.plot(y, y-modeleReg.predict(X),'.',markersize=0.5)
plt.plot(a,a,color='r')
plt.title('Résidus en fonction de la valeur observées')
plt.xlabel("Valeur observées")
plt.ylabel('Résidus')


In [ ]:
plt.figure(figsize=(10,10))
sns.histplot(y-modeleReg.predict(X),kde=True,kde_kws={'bw_adjust':2})
plt.xlim(-200,200)
plt.title('distribution des résidus')
#plt.xlim([-50,-30])

##### Formule à partir des données métiers

la théorie nous dis kJ= 37,2 x Lipide + 16,7 x Glucide + 16,7 x Proteines + 8.36 x Fibres

In [ ]:
df_test['energie_calculée']=(37.6*df_test['fat_100g']+
                             16.7*df_test['proteins_100g']+
                             16.7*df_test['carbohydrates_100g']
                            )

##### Traitement des erreurs d'unités

Selectionnons les individus qui présente un coefficient de 4.18 à plus ou moins 25%

In [ ]:
df_units=df_test[(df_test['energy_100g'] < df_test['energie_calculée']/4.18*1.15) & 
                        (df_test['energy_100g'] > df_test['energie_calculée']/4.18*0.85) ]

In [ ]:
plt.plot(df_units['energy_100g'], df_units['energie_calculée'],'.',markersize=1)
plt.xlabel('Valeurs observées')
plt.ylabel('Valeurs calculées')
plt.xlim([0,3000])
plt.ylim([0,3000])

In [ ]:
df_units.shape

Multiplions par 4.18 pour remettre la valeurs dans la bonne unités

In [ ]:
df_test.loc[df_units.index,"energy_100g"]=df_test.loc[df_units.index,"energy_100g"]*4.18

##### Traitement des outliers

définissons outliers par rapport a une erreur entre la valeur théorique et observé de 25%

In [ ]:
df_outliers=df_test[(df_test['energy_100g'] > df_test['energie_calculée']*1.25) | 
                        (df_test['energy_100g'] < df_test['energie_calculée']*0.75) ]

In [ ]:
df_outliers.shape

In [ ]:
df_test.shape

In [ ]:
df_outliers.describe()

In [ ]:
plt.plot(df_outliers['energy_100g'], df_outliers['energie_calculée'],'.',markersize=1)
plt.xlabel('Valeurs observées')
plt.ylabel('Valeurs calculées')
plt.xlim([0,1000])
plt.ylim([0,1000])

In [ ]:
df_test.describe()

In [ ]:
fig, ax = plt.subplots(5,3)
fig.set_size_inches(30, 20)
i=0
for col in df_test.describe().columns:
    
    #sns.kdeplot(x=df_impute.loc[i_mod][col],ax=ax[i//2][i%2],cmap="Reds", shade=True,label='valeurs imputées')
    sns.kdeplot(x=df_outliers[col],ax=ax[i//3][i%3],color='b', shade=True,label='Outliers')
    sns.kdeplot(x=df_test[col],ax=ax[i//3][i%3],color='r', shade=True, label='Dataset')
    fig.legend()  
    i+=1
plt.tight_layout()

In [ ]:
sns.countplot(data=df_outliers,x='nutrition_grade_fr',order=['a','b','c','d','e'])

In [ ]:
sns.countplot(data=df_test,x='nutrition_grade_fr',order=['a','b','c','d','e'])

In [ ]:
sns.countplot(data=df_impute,x='nutrition_grade_fr',order=['a','b','c','d','e'])

In [ ]:
df_test=df_test.drop(index=df_outliers.index)

In [ ]:
df_test=df_test.drop(columns=['energie_calculée'])

####  Imputation des données

In [ ]:
def imputation(data,table): #return the index of the imputed values
    i=0
    index=[]
    counter=0
    for i in data.index:
        #print(f'i={i}')
        #print(f'mmissing values {data.loc[i][list(table.keys())].isna().sum()}')
        if data.loc[i][list(table.keys())].isna().sum()==1:# check if there is one and only missing value
            for col_to_fill in table.keys():
                #print(col_to_fill)
                if np.isnan(data.loc[i][col_to_fill]):#identify the column of the missing value
                    #print(f'missing value in columns {col_to_fill}')
                    data.loc[i,col_to_fill]=0
                    
                    cols=list(table.keys()) # creates the dictionnary of the other columns
                    cols.remove(col_to_fill)                   
                    
                    for col in cols: #calculate the theoretical value
                        data.loc[i,col_to_fill] = data.loc[i,col_to_fill]-data.loc[i,col]*table[col]/table[col_to_fill]
                        
                    if data.loc[i,col_to_fill]>100:
                        data.loc[i,col_to_fill]=np.nan
                            
                    elif data.loc[i,col_to_fill]<0:
                            data.loc[i,col_to_fill]=np.nan
                    else:                     
                        #print(f'filled data [{i},{col_to_fill}] with {data.loc[i,col_to_fill]}')
                        index.append(i)
                        counter+=1
    print(f'job done modified {counter} values')
    return index
      

###### Sur un echantillon pour test

In [ ]:
df_test_sample=df_test.sample(20000)
df_test_sample_impute=df_test_sample.copy()


In [ ]:
tab={'fat_100g': 37.2,
     'proteins_100g': 16.7,
     'carbohydrates_100g': 16.7,
     'fiber_100g': 8.36,
     'energy_100g': -1,
    }

In [ ]:
i_mod=imputation(df_test_sample_impute,tab)

Effet sur les variables

In [ ]:
cols=['fat_100g',
     'proteins_100g',
     'carbohydrates_100g',
     'fiber_100g',
     'energy_100g']

In [ ]:
fig, ax = plt.subplots(3,2)
fig.set_size_inches(30, 20)
i=0
for col in cols:
    T=pd.concat([df_test_sample_impute[col],df_test_sample[col]],axis=1)
    T.columns=['imputed','not imputed']
    sns.boxplot(x="variable", y="value", data=pd.melt(T), ax=ax[i//2][i%2])
    plt.title(f'{col}')
    i+=1

In [ ]:
fig, ax = plt.subplots(3,2)
fig.set_size_inches(30, 20)
i=0
for col in cols:
    
    #sns.kdeplot(x=df_impute.loc[i_mod][col],ax=ax[i//2][i%2],cmap="Reds", shade=True,label='valeurs imputées')
    sns.kdeplot(x=df_test_sample_impute.loc[i_mod,col],ax=ax[i//2][i%2],color='b', shade=True,label='dataset imputé')
    sns.kdeplot(x=df_test_sample[col],ax=ax[i//2][i%2],color='r', shade=True, label='dataset avant imputation')
    fig.legend()  
    i+=1
plt.tight_layout()

###### Application à toutes les données

Imputation

In [ ]:
df_impute=df_test.copy()

In [ ]:
i_mod=imputation(df_impute,tab)

In [ ]:
df_impute.shape

Effet sur les variables

In [ ]:
cols=['energy_100g', 'fat_100g','carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g']

In [ ]:
fig, ax = plt.subplots(3,2)
fig.set_size_inches(30, 20)
i=0
for col in cols:
    
    #sns.kdeplot(x=df_impute.loc[i_mod][col],ax=ax[i//2][i%2],cmap="Reds", shade=True,label='valeurs imputées')
    sns.kdeplot(x=df_impute.loc[i_mod,col],ax=ax[i//2][i%2],color='b', shade=True,label='dataset imputé')
    sns.kdeplot(x=df_test[col],ax=ax[i//2][i%2],color='r', shade=True, label='dataset avant imputation')
    fig.legend()  
    i+=1
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(3,2)
fig.set_size_inches(30, 20)
i=0
for col in cols:
    T=pd.concat([df_impute[col],df_test[col]],axis=1)
    T.columns=['imputed','not imputed']
    sns.boxplot(x="variable", y="value", data=pd.melt(T), ax=ax[i//2][i%2])
    plt.title(f'{col}')
    i+=1

In [ ]:
df_impute.to_pickle("data/data_impute")

In [ ]:
df_impute=pd.read_pickle("data/data_impute")

### ACP

In [ ]:
df_impute=pd.read_pickle("data/data_impute")

In [ ]:
df_impute.shape

In [ ]:
df_acp=df_impute.copy()

Selection des variables pour l'ACP

In [ ]:
cols=['carbohydrates_100g',
       'saturated-fat_100g', 'sugars_100g', 'proteins_100g',
       'fiber_100g']

In [ ]:
i=0
colors=sns.color_palette("Set3", 14)
plt.figure(figsize=(15,15))
for col in cols:
    i+=1
    plt.subplot(4,3,i)
    sns.boxplot(x=df_impute[col],color=colors[i])

In [ ]:
#df_acp=df_acp[cols]

In [ ]:
df_acp=df_acp.dropna(subset=cols)

In [ ]:
df_acp.shape

In [ ]:
df_acp[cols].quantile(0.98)

In [ ]:
df_acp_core=df_acp.copy()

In [ ]:
for col in cols:
    df_acp_core=df_acp_core.drop(index=df_acp_core[df_acp_core[col]>=df_acp_core[col].quantile(0.98)].index)

In [ ]:
df_acp_core.shape

In [ ]:
i=0
colors=sns.color_palette("Set3", 14)
plt.figure(figsize=(10,15))
for col in cols:
    i+=1
    plt.subplot(4,2,i)
    sns.boxplot(x=df_acp_core[col],color=colors[i])

In [ ]:
df_ext=df_acp.drop(index=df_acp_core.index)

In [ ]:
df_ext.shape

In [ ]:
i=0
colors=sns.color_palette("Set3", 14)
plt.figure(figsize=(10,15))
for col in cols:
    i+=1
    plt.subplot(4,2,i)
    sns.boxplot(x=df_ext[col],color=colors[i])

In [ ]:
i=0
plt.figure(figsize=(10,15))
for col in cols:
    i+=1
    plt.subplot(4,2,i)
    data=df_ext[df_ext[col]>df_acp[col].quantile(0.95)]
    sns.countplot(data=data,x='nutrition_grade_fr',order=['a','b','c','d','e'])
    plt.title(col)
plt.tight_layout()

In [ ]:
df_acp=df_acp_core[cols]

#### Preprocessing des données

##### Standard Scaler

In [ ]:
from sklearn import preprocessing

In [ ]:
X=df_acp.values
std_scale = preprocessing.StandardScaler().fit(X)
X_scaled_s = std_scale.transform(X)

for i in range(X.shape[1]):
   sns.kdeplot(X_scaled_s[:,i])


In [ ]:
X_scaled_s.mean(axis=0)

##### Min-max Scaler


In [ ]:
X=df_acp.values
m_scale = preprocessing.MinMaxScaler().fit(X)
X_scaled_m = m_scale.transform(X)
for i in range(X_scaled_m.shape[1]):
    X_scaled_m[:,i]-=X_scaled_m[:,i].mean()

for i in range(X.shape[1]):
   sns.kdeplot(X_scaled_m[:,i])

In [ ]:
X_scaled_m.mean(axis=0)

##### Robust Scaler


In [ ]:
X=df_acp.values
r_scale = preprocessing.RobustScaler().fit(X)
X_scaled_r = r_scale.transform(X)
for i in range(X_scaled_m.shape[1]):
    X_scaled_r[:,i]-=X_scaled_r[:,i].mean()

for i in range(X.shape[1]):
   sns.kdeplot(X_scaled_r[:,i])

In [ ]:
X_scaled_r.mean(axis=0)

#### Calcul des composantes principales

In [ ]:
from sklearn import decomposition

In [ ]:
pca_s = decomposition.PCA(n_components=5)
pca_s.fit(X_scaled_s)

pca_m = decomposition.PCA(n_components=5)
pca_m.fit(X_scaled_m)

pca_r = decomposition.PCA(n_components=5)
pca_r.fit(X_scaled_r)

#### Eboulis des valeurs propres

In [ ]:
print(' '*20,'Axes principaux',' '*20)
print('-'*18,'Variance expliquée','-'*18)
print(pca_s.explained_variance_ratio_)
print('-'*15,'Variance expliquée cumulée','-'*15)
print(pca_s.explained_variance_ratio_.cumsum())



In [ ]:
print('min-max scaler','-'*40)
print(pca_m.explained_variance_ratio_)
print(pca_m.explained_variance_ratio_.cumsum())
print(pca_m.explained_variance_ratio_.sum())
print('robust scaler','-'*40)
print(pca_r.explained_variance_ratio_)
print(pca_r.explained_variance_ratio_.cumsum())
print(pca_r.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(15,5))

val=pca_s.explained_variance_ratio_

plt.subplot(1,3,1)
plt.bar(np.arange(len(val))+1,val)
plt.plot(np.arange(len(val))+1,val.cumsum(),c='purple')
plt.title('Variance expliquée - Standard Scaler')

val=pca_m.explained_variance_ratio_

plt.subplot(1,3,2)
plt.bar(np.arange(len(val))+1,val)
plt.plot(np.arange(len(val))+1,val.cumsum(),c='purple')
plt.title('Variance expliquée - Min-max Scaler')

val=pca_m.explained_variance_ratio_

plt.subplot(1,3,3)
plt.bar(np.arange(len(val))+1,val)
plt.plot(np.arange(len(val))+1,val.cumsum(),c='purple')
plt.title('Variance expliquée - Robust Scaler')

#### Projection des individus

In [ ]:
def plot_ind(X,pca,titre):

    #projection de X sur les composantes principales
    X_projected = pca.transform(X)


    fig = plt.figure(figsize=(16,8))
    plt.title(titre)

    plt.subplot(1,2,1)
    sns.scatterplot(x=X_projected[:,0], y=X_projected[:,1],s=2)
    plt.title(titre+'- premier plan')
    plt.xlabel(f'PC1 - {round(pca.explained_variance_ratio_[0],3)*100}%')
    plt.ylabel(f'PC2 - {round(pca.explained_variance_ratio_[1],3)*100}%')
    
    plt.subplot(1,2,2)
    sns.scatterplot(x=X_projected[:,1], y=X_projected[:,2],s=2)
    plt.xlabel(f'PC2 - {round(pca.explained_variance_ratio_[1],3)*100}%')
    plt.ylabel(f'PC3 - {round(pca.explained_variance_ratio_[2],3)*100}%')

    plt.title(titre+'- second plan')
    plt.show()

In [ ]:
plot_ind(X_scaled_s,pca_s,'Projection des individus - scaler standard')

In [ ]:
plot_ind(X_scaled_m,pca_m,'Projection des individus - Minmax scaler')

In [ ]:
plot_ind(X_scaled_r,pca_r,'Projection des individus - Robust scaler')

#### Projection de tous les individus (incluant outliers)

In [ ]:
df_impute.shape

In [ ]:
df_tot_acp=df_impute[cols]

In [ ]:
df_tot_acp=df_tot_acp.dropna()

In [ ]:
df_tot_acp.shape

In [ ]:
X_scaled_tot_s = std_scale.transform(df_tot_acp.values)

In [ ]:
X_scaled_tot_m = m_scale.transform(df_tot_acp.values)
for i in range(X_scaled_m.shape[1]):
    X_scaled_tot_m[:,i]-=X_scaled_m[:,i].mean()

In [ ]:
X_scaled_tot_r = r_scale.transform(df_tot_acp.values)
for i in range(X_scaled_r.shape[1]):
    X_scaled_tot_r[:,i]-=X_scaled_tot_r[:,i].mean()

In [ ]:
plot_ind(X_scaled_tot_s,pca_s,'Projection des individus - scaler standard')

In [ ]:
plot_ind(X_scaled_tot_m,pca_m,'Projection des individus - Minmax scaler')

In [ ]:
plot_ind(X_scaled_tot_r,pca_r,'Projection des individus - Robust scaler')

#### Contribution de chaque variables aux composantes principales

In [ ]:
def plot_variables(pca,titre):
    pcs = pca.components_
    plt.figure(figsize=(16,8))
    
    plt.subplot(1,2,1)
    for i, (x,y) in enumerate(zip(pcs[0,:], pcs[1,:])):
        plt.plot([0,x],[0,y],'b--')
        plt.text(x,y,df_acp.columns[i], fontsize='12')
        plt.title(titre+' - premier plan')
        plt.xlabel(f'PC1 - {round(pca.explained_variance_ratio_[0],3)*100}%')
        plt.ylabel(f'PC2 - {round(pca.explained_variance_ratio_[1],3)*100}%')
        
    # Afficher une ligne horizontale y=0
    plt.plot([-0.7, 0.7], [0, 0], color='grey', ls='--')
    # Afficher une ligne verticale x=0
    plt.plot([0, 0], [-0.7, 0.7], color='grey', ls='--')

    plt.subplot(1,2,2)
    for i, (x,y) in enumerate(zip(pcs[1,:], pcs[2,:])):
        plt.plot([0,x],[0,y],'b--')
        plt.text(x,y,df_acp.columns[i], fontsize='12')
        plt.title(titre+' - deuxième plan')
        plt.xlabel(f'PC2 - {round(pca.explained_variance_ratio_[1],3)*100}%')
        plt.ylabel(f'PC3 - {round(pca.explained_variance_ratio_[2],3)*100}%')
    
     # Afficher une ligne horizontale y=0
    plt.plot([-0.7, 0.7], [0, 0], color='grey', ls='--')
    # Afficher une ligne verticale x=0
    plt.plot([0, 0], [-0.7, 0.7], color='grey', ls='--')


In [ ]:
pca_s.components_

In [ ]:
plot_variables(pca_s,'composition des composantes - standard scaler')

In [ ]:
plot_variables(pca_m,'composition des composantes - Minmax scaler')

In [ ]:
plot_variables(pca_r,'composition des composantes - robust scaler')

#### Visualisation 3D

In [ ]:
import plotly.express as px


In [ ]:
X_projected = pca_s.transform(X_scaled_s)

In [ ]:
def plot_3D(data,pca):
    X_projected = pca.transform(data)
    fig=px.scatter_3d(X_projected,
              x=X_projected[:,0],
              y=X_projected[:,1],
              z=X_projected[:,2],
              opacity=0.1,
              )
    fig.update_traces(marker=dict(size=2,
                                  #line=dict(width=0.05)
                                 ),
                      selector=dict(mode='markers')
                     )
    fig.show()

In [ ]:
plot_3D(X_scaled_s,pca_s)

In [ ]:
plot_3D(X_scaled_m,pca_m)

In [ ]:
plot_3D(X_scaled_r,pca_r)

#### Capacité à disciminer la nutrition_grade_fr

In [ ]:
c=df_impute.loc[df_acp.index]['nutrition_grade_fr']

In [ ]:
c_tot=df_impute.loc[df_tot_acp.index]['nutrition_grade_fr']

In [ ]:
c_tot.shape

In [ ]:
c.shape

In [ ]:
c=c.fillna('0')
c_tot=c_tot.fillna('0')

In [ ]:
X_projected = pca_s.transform(X_scaled_s)

fig = px.scatter_3d(
              x=X_projected[:,0],
              y=X_projected[:,1],
              z=X_projected[:,2],
              color=c,
              color_discrete_map={
                  'a':'green',
                  'b':'blue',
                  'c':'yellow',
                  'd':'orange',
                  'e':'red',
                  '0':'white'
              },
              category_orders={"color":['0','a','b','c','d','e']}
)

fig.update_traces(marker=dict(size=2,
                              ),
                  selector=dict(mode='markers'))

fig.show()

In [ ]:
plt.figure(figsize=(10,10))
sns.scatterplot(x=X_projected[:,0],
                y=X_projected[:,1],
                s=2,
                palette={'a':'green',
                         'b':'blue',
                         'c':'yellow',
                         'd':'orange',
                         'e':'red',
                         '0':'white'
                },
                hue=c,
                hue_order=['0','a','b','c','d','e']
)
plt.title('ACP - premier plan')
plt.xlabel('PC1')
plt.ylabel('PC2')

In [ ]:
plt.figure(figsize=(10,10))
sns.scatterplot(x=X_projected[:,1],
                y=X_projected[:,2],
                s=2,
                palette={'a':'green',
                         'b':'blue',
                         'c':'yellow',
                         'd':'orange',
                         'e':'red',
                         '0':'white'
                },
                hue=c,
                hue_order=['0','a','b','c','d','e']
)
plt.title('ACP - second plan')
plt.xlabel('PC2')
plt.ylabel('PC3')

### AFD sur nutrition_grade_fr

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
method = LinearDiscriminantAnalysis() 
lda = method.fit(X_scaled_m, c)
X_r2 = lda.transform(X_scaled_m)

In [ ]:
plt.figure(figsize=(10,10))
sns.scatterplot(x=X_r2[:,0],
                y=X_r2[:,1],
                s=2,
                palette={'a':'green',
                         'b':'blue',
                         'c':'yellow',
                         'd':'orange',
                         'e':'red',
                         '0':'white'
                },
                hue=c,
                hue_order=['0','a','b','c','d','e']
)
plt.title('AFD - premier plan')
plt.xlabel('PC1')
plt.ylabel('PC2')


In [ ]:
plt.figure(figsize=(10,10))
sns.scatterplot(x=X_r2[:,1],
                y=X_r2[:,2],
                s=2,
                palette={'a':'green',
                         'b':'blue',
                         'c':'yellow',
                         'd':'orange',
                         'e':'red',
                         '0':'white'
                },
                hue=c,
                hue_order=['0','a','b','c','d','e']
)
plt.title('AFD - second plan')
plt.xlabel('PC2')
plt.ylabel('PC3')



In [ ]:
X_projected = X_r2

fig = px.scatter_3d(
              x=X_projected[:,0],
              y=X_projected[:,1],
              z=X_projected[:,2],
              color=c,
              color_discrete_map={
                  'a':'green',
                  'b':'blue',
                  'c':'yellow',
                  'd':'orange',
                  'e':'red',
                  '0':'white'
              },
              category_orders={"color":['0','a','b','c','d','e']}
)

fig.update_traces(marker=dict(size=3,
                              ),
                  selector=dict(mode='markers'))

fig.show()

In [ ]:
def plot_coef(method,titre):
    pcs = method.scalings_
    plt.figure(figsize=(16,8))
    
    plt.subplot(1,2,1)
    for i, (x,y) in enumerate(zip(pcs[:,0], pcs[:,1])):
        plt.plot([0,x],[0,y],'b--')
        plt.text(x,y,df_acp.columns[i], fontsize='12')
        plt.title(titre+' - premier plan')
        plt.xlabel('LD1')
        plt.xlabel('LD2')
        


    plt.subplot(1,2,2)
    for i, (x,y) in enumerate(zip(pcs[:,1], pcs[:,2])):
        plt.plot([0,x],[0,y],'b--')
        plt.text(x,y,df_acp.columns[i], fontsize='12')
        plt.title(titre+' - deuxième plan')
        plt.xlabel('LD2')
        plt.xlabel('LD3')
    
 


In [ ]:
plot_coef(method,'composition des composantes AFD')

## 2.2 Variables qualitatives

In [ ]:
df_test=pd.read_pickle("data/data_impute")

In [ ]:
for col in df_test.select_dtypes('string'):
    print (f'{col:>20} {len(df_test[col].unique()):>20} {df_test[col].isna().sum(axis = 0)/df_test.shape[0]:>20.3f}')
    print('-'*70)

### Variable nutrition_grade_fr

In [ ]:
sns.countplot(data=df_test,x='nutrition_grade_fr',order=['a','b','c','d','e'])

#### ANOVA

In [ ]:
fig, ax = plt.subplots(7,2)
fig.set_size_inches(10, 25)
i=0
for col in df_test.describe().columns:
   sns.boxplot(x='nutrition_grade_fr',
               y=col,
               data=df_test,
               palette='rainbow',
               order=['a','b','c','d','e'],
               showfliers=False,
               ax=ax[i//2][i%2]
               )
   i+=1

In [ ]:
from scipy.stats import f_oneway

In [ ]:
def anova_multiple_p(data,cols):
    R=[]
    for col in cols:
        df_test=data.dropna(subset=cols)
        p=f_oneway(df_test[df_test.nutrition_grade_fr=='a'][col],
                   df_test[df_test.nutrition_grade_fr=='b'][col],
                   df_test[df_test.nutrition_grade_fr=='c'][col],
                   df_test[df_test.nutrition_grade_fr=='d'][col],
                   df_test[df_test.nutrition_grade_fr=='e'][col]
        )[1]
        R.append(p)
    return R

In [ ]:
anova_multiple_p(df_test,['saturated-fat_100g',
                          'proteins_100g',
                          'carbohydrates_100g',
                          'fiber_100g',
                          'energy_100g'
                         ]
                )

In [ ]:
anova=[]
for a in np.arange(50,600,25):
    for i in range(20):
        df=df_test.sample(a)
        anova.append([a]+anova_multiple_p(df,['saturated-fat_100g',
                                          'proteins_100g',
                                          'carbohydrates_100g',
                                          'fiber_100g',
                                          'energy_100g'
                                         ]
                                     )
                )
        
        

In [ ]:
anova=pd.DataFrame(anova,columns=['samples','saturated-fat_100g',
                              'proteins_100g',
                              'carbohydrates_100g',
                              'fiber_100g',
                              'energy_100g'
                             ]
                  )

In [ ]:
anova_mean=anova.pivot_table(index=['samples'])

In [ ]:
anova_mean

In [ ]:
anova_mean.shape

In [ ]:

fig, ax = plt.subplots(figsize=(7,7))
for i in range(anova_mean.shape[1]):
   plt.plot(anova_mean.iloc[:,i],label=anova_mean.columns[i])

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc=5)
plt.xlabel('taille de l\'échantillon')
plt.ylabel('moyenne des p-value')
plt.title('Moyenne des p-value du test oneway Anova sur 20 échantillonnages')
plt.show()